# Working with JSON

This module covers JSON as a data format and Python's `json` module for reading and writing it. Readers are assumed to be comfortable with Python syntax and at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on what JSON looks like, how Python loads and walks it, and the patterns that come up when configuration, exports, and audit files in a tm1py workflow are written as JSON.

JSON shows up in the files around a tm1py script far more than inside it. tm1py methods return Python objects, so the wire format is hidden. What is not hidden is the connection config the script reads at startup, the audit record it writes after a run, the export it hands to a downstream system, and the master data file an upstream system delivers each morning. Most of what a TM1 user needs from JSON is the ability to load these files, walk what comes back, and write the structure out again.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. What a JSON file looks like
2. Where JSON shows up in TM1 work
3. Loading a JSON file
4. Walking the parsed structure
5. Modifying and writing JSON back
6. Building JSON from a Python value
7. The Python and JSON type mapping
8. Pretty printing and stable output
9. Values that need help: datetime, Decimal, dataclass
10. Customizing parsing with `object_hook`
11. JSON Lines for append friendly logs
12. Real world design principles
13. Common mistakes

---

## 1. What a JSON file looks like

JSON (JavaScript Object Notation) is a plain text format for nested data. A JSON file is a single value: an object (curly braces), an array (square brackets), a string, a number, `true`, `false`, or `null`. Most files begin with `{` or `[` and contain everything else inside.

Here is a config file for a tm1py job that loads two cubes from a TM1 server and writes an audit record:

```json
{
  "server": {
    "address": "tm1.example.com",
    "port": 8001,
    "ssl": true,
    "user": "admin"
  },
  "cubes": [
    {"name": "Sales Plan", "view": "Plan Input"},
    {"name": "General Ledger", "view": "Actuals"}
  ],
  "output": {
    "directory": "/var/lib/tm1py/exports",
    "format": "csv"
  },
  "notify": null
}
```

Three things are worth noticing. First, every key is a string in double quotes; single quotes are not allowed in JSON. Second, `true`, `false`, and `null` are bare lowercase words, not strings. Third, the structure nests freely: an object can contain arrays, an array can contain objects, an object's value can be another object, and so on to any depth.

This file is the running example for the rest of the module. The same shape appears in real TM1 projects as a `config.json` next to the script, in test fixtures, and in handoffs between systems.

## 2. Where JSON shows up in TM1 work

A tm1py script lives in a small ecosystem of supporting files, most of which are JSON:

- **Connection and job config.** Address, port, credentials, the list of cubes or views to process, output paths. Splitting these out of the script makes the script reusable across environments (dev, test, prod) by swapping the config file.
- **Imports.** Upstream systems (HR, CRM, ERP) often deliver master data and transactions as JSON. Loading them means parsing into Python first, then writing into TM1 with `tm1.elements.create`, `tm1.cells.write_values`, or a TI process.
- **Exports.** A nightly job that exports a slice of the Sales Plan cube to a downstream forecasting tool typically writes JSON or JSON Lines (Topic 11), since both are easier to consume from any language than CSV with a custom delimiter.
- **Audit and run logs.** A record per job run that captures what was loaded, how long it took, and whether it succeeded. JSON Lines is the right format for these.
- **Test fixtures.** A captured response saved to disk and replayed in a unit test so the test does not need a live TM1 server.

tm1py itself speaks JSON over HTTPS to the TM1 REST API, but the library hides that: methods return dicts, lists, and tm1py objects, not raw JSON. The JSON in this module is the JSON the TM1 user reads and writes directly, in the files around their script.

## 3. Loading a JSON file

`json.load` reads from an open file object and returns the parsed Python value. The file should be opened in text mode with an explicit UTF 8 encoding:

In [ ]:
import json
from pathlib import Path

with Path("config.json").open("r", encoding="utf-8") as f:
    config: dict[str, object] = json.load(f)

type(config)
# <class 'dict'>

The whole file becomes one Python value. The outer braces in the JSON file map to a `dict`; an outer `[` would map to a `list`. From here on, `config` is a normal Python dict and behaves like one.

`json.loads` (with the `s` for "string") is the variant that parses a string already in memory, which is useful when the JSON arrived from somewhere other than a file:

In [ ]:
text: str = '{"address": "tm1.example.com", "port": 8001}'
server: dict[str, object] = json.loads(text)
# {'address': 'tm1.example.com', 'port': 8001}

Always pass `encoding="utf-8"` when opening the file. The file encoding default depends on the host operating system, and a config file written on Windows under cp1252 will not round trip through a Linux container set to UTF 8. JSON itself mandates Unicode, and UTF 8 is the only sensible choice on disk.

## 4. Walking the parsed structure

A parsed JSON document is a tree of Python dicts, lists, strings, numbers, booleans, and `None`. Walking it uses ordinary Python: `[]` indexing for both dicts and lists, `for` loops for arrays, `.get()` for keys that may be missing.

Picking values out of the config from Topic 1:

In [ ]:
config["server"]["address"]
# 'tm1.example.com'

config["server"]["ssl"]
# True

config["cubes"][0]["name"]
# 'Sales Plan'

len(config["cubes"])
# 2

Iterating over the array of cubes:

In [ ]:
for cube in config["cubes"]:
    print(cube["name"], "->", cube["view"])
# Sales Plan -> Plan Input
# General Ledger -> Actuals

For optional keys, `dict.get` is safer than `[]` because it returns `None` (or a chosen default) instead of raising `KeyError`:

In [ ]:
config.get("notify")
# None

config.get("retry_count", 3)
# 3        (the key is absent, so the default is returned)

A common shape in TM1 exports is a list of records where one field is the key and another is the value. Building a name to value lookup from such a list takes a single dict comprehension:

In [ ]:
view_by_cube: dict[str, str] = {cube["name"]: cube["view"] for cube in config["cubes"]}
# {'Sales Plan': 'Plan Input', 'General Ledger': 'Actuals'}

The structure is recursive; a script that walks an arbitrary configuration tree is just a recursive function over `dict` and `list`. There is nothing JSON specific about the access. Once the file is loaded, the JSON is gone and only Python is left.

## 5. Modifying and writing JSON back

The dict returned by `json.load` is mutable. Editing it and writing the result back is the usual flow when one tool produces a config and another updates it:

In [ ]:
config["server"]["port"] = 8010
config["cubes"].append({"name": "Headcount", "view": "Plan"})

with Path("config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

`json.dump` writes a Python value to an open file. Pass `indent=2` (or any positive integer) for human readable output; without it, the whole file is one line. The argument order is `(value, file)`.

For round trips that should produce byte identical files (config in version control, golden files in tests), pass `sort_keys=True` so the keys come out in lexicographic order regardless of insertion order, and `ensure_ascii=False` so non ASCII characters appear as themselves rather than as `\uXXXX` escapes:

In [ ]:
with Path("config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, sort_keys=True, ensure_ascii=False)

A cube called `"Forderungen"` and a region called `"München"` are common in real TM1 models. With `ensure_ascii=False` they appear in the file as written; with the default they come out as `"München"`, which loads back to the same string but is unreadable in a text editor.

## 6. Building JSON from a Python value

`json.dumps` (with the `s`) returns the JSON text as a string instead of writing to a file. Useful when the destination is something other than a file: an HTTP body, a log message, a database column.

In [ ]:
audit: dict[str, object] = {
    "run_id": "r-001",
    "started_at": "2026-04-30T14:22:00",
    "cube": "Sales Plan",
    "rows_loaded": 12_400,
    "ok": True,
}

line: str = json.dumps(audit)
# '{"run_id": "r-001", "started_at": "2026-04-30T14:22:00", "cube": "Sales Plan", "rows_loaded": 12400, "ok": true}'

Note how `True` becomes the bare word `true` and the underscore in `12_400` is not in the output. JSON has no underscore separator for digits and no Python style booleans; the encoder translates each into the JSON spelling. For compact output (one record per line, no spaces between separators), pass `separators`:

In [ ]:
json.dumps(audit, separators=(",", ":"))
# '{"run_id":"r-001","started_at":"2026-04-30T14:22:00","cube":"Sales Plan","rows_loaded":12400,"ok":true}'

The encoder accepts dicts, lists, tuples, strings, ints, floats, booleans, and `None`. Anything else raises `TypeError`. The next topic gives the complete mapping; Topic 9 covers what to do when a value is one of the unsupported types.

## 7. The Python and JSON type mapping

JSON has six value types. Python's `json` module maps each one to a built in Python type:

| JSON     | Python                |
|----------|-----------------------|
| string   | `str`                 |
| number   | `int` or `float`      |
| boolean  | `bool`                |
| null     | `None`                |
| array    | `list`                |
| object   | `dict` (string keys)  |

The mapping is asymmetric. JSON has fewer types than Python, and the round trip is lossy in a few specific places:

- **Tuples become lists.** `json.dumps((1, 2))` produces `'[1, 2]'`, and parsing that back gives `[1, 2]`. Code that relied on `isinstance(value, tuple)` will break after a JSON round trip.
- **Dict keys become strings.** `json.dumps({2026: "current"})` produces `'{"2026": "current"}'`, and parsing it back gives `{"2026": "current"}` with the key as a string. A dict keyed by `int` does not survive.
- **Dict keys that are not strings, ints, floats, bools, or `None` raise `TypeError`.** A `tuple` key, which is the natural shape of a tm1py cellset, cannot be serialized at all without restructuring (see Topic 13).
- **`int` and `float` are distinct in Python but not in JSON.** `1` stays `1` and `1.0` stays `1.0` on output; on input, the parser picks `int` if the literal has no decimal point or exponent, otherwise `float`.
- **Several common Python types have no JSON equivalent at all:** `set`, `frozenset`, `bytes`, `datetime`, `date`, `Decimal`, dataclass instances, enums. Topic 9 covers how to handle them.

For most TM1 work the mapping just works: configs are nested dicts and lists of strings and numbers, exports are records with string and number fields, audit logs are flat dicts. The asymmetric cases come up when the data includes a timestamp, a precise decimal amount, or a cellset whose keys are tuples.

## 8. Pretty printing and stable output

Topics 5 and 6 introduced `indent`, `sort_keys`, and `ensure_ascii`. Together they cover almost every formatting need:

In [ ]:
config = {"port": 8001, "address": "tm1.example.com", "ssl": True}

print(json.dumps(config))
# {"port": 8001, "address": "tm1.example.com", "ssl": true}

print(json.dumps(config, indent=2))
# {
#   "port": 8001,
#   "address": "tm1.example.com",
#   "ssl": true
# }

print(json.dumps(config, indent=2, sort_keys=True))
# {
#   "address": "tm1.example.com",
#   "port": 8001,
#   "ssl": true
# }

The choice of options depends on the consumer of the file:

- **Files committed to version control:** `indent=2, sort_keys=True, ensure_ascii=False`. Diffs stay small and meaningful, and accidental key reorderings do not show up as spurious changes in code review.
- **Files written and read by the same script:** the defaults are fine. No human reads them.
- **Records on a wire or in a single line log:** `separators=(",", ":")`. Strips the spaces that the default puts after `,` and `:`, which can shave a noticeable percentage off the size of a high volume log.

`indent` and `separators` interact: setting `indent` already changes the default separators to `(",", ": ")` (with a trailing space after the colon, which is part of pretty output). Setting `separators` explicitly overrides that.

## 9. Values that need help: datetime, Decimal, dataclass

A first attempt at serializing an audit record with a real timestamp and a precise amount fails:

In [ ]:
import json
from datetime import datetime
from decimal import Decimal

audit = {
    "run_at": datetime(2026, 4, 30, 14, 22, 0),
    "amount": Decimal("120000.50"),
}
json.dumps(audit)
# TypeError: Object of type datetime is not JSON serializable

`json.dumps` accepts a `default` callable that runs whenever the encoder meets a type it does not know. The callable receives the value and must return something the encoder does know how to handle (typically a string, list, or dict):

In [ ]:
from dataclasses import is_dataclass, asdict

def to_json(value: object) -> object:
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)            # preserve precision; a float would drift
    if isinstance(value, set):
        return sorted(value)         # arrays preserve order, sets do not
    if is_dataclass(value):
        return asdict(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")

print(json.dumps(audit, default=to_json, indent=2))
# {
#   "run_at": "2026-04-30T14:22:00",
#   "amount": "120000.50"
# }

Three points matter. First, the function is called once per unsupported value, so it must be total over the types the project actually emits. Second, the final `raise` is essential: a silent fallback to `str(value)` works for everything because every Python object has a `__str__`, but it produces silently wrong JSON for types the project did not intend to support. Third, `Decimal` is converted to a string so the precision is preserved on the way out; converting it to `float` would defeat the reason for using `Decimal` in the first place.

## 10. Customizing parsing with `object_hook`

`json.load` and `json.loads` accept an `object_hook` callable that runs on every JSON object after it is parsed into a `dict`. The return value replaces the dict in the result. It is the symmetric counterpart to `default`: instead of teaching the encoder about extra types, it teaches the decoder to produce richer ones than plain dicts.

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class CubeJob:
    name: str
    view: str

def from_json(d: dict[str, object]) -> object:
    if set(d.keys()) == {"name", "view"}:
        return CubeJob(name=d["name"], view=d["view"])
    return d

with Path("config.json").open("r", encoding="utf-8") as f:
    config = json.load(f, object_hook=from_json)

config["cubes"]
# [CubeJob(name='Sales Plan', view='Plan Input'),
#  CubeJob(name='General Ledger', view='Actuals')]

The hook fires bottom up: nested objects are converted before their containers see them. The function must handle every shape the file might have, including ones it does not recognize, by returning the dict unchanged. Returning `None` for unrecognized shapes drops them from the result, which is almost never what is wanted.

`object_hook` is the right tool when the project repeatedly walks the same shape and would benefit from real types: autocomplete, type checking, equality, frozen dataclasses for safety. For one off scripts the plain dicts from Topic 4 are simpler.

## 11. JSON Lines for append friendly logs

JSON Lines (also written `.jsonl` or NDJSON) is a convention, not a separate format: one JSON value per line, separated by `\n`. It is the right shape whenever the data is a sequence of independent records, written incrementally, large enough that loading the whole file at once is undesirable, or appended to over time:

In [ ]:
import json
from pathlib import Path

audit_path = Path("audit.jsonl")

new_record: dict[str, object] = {
    "run_id": "r-042",
    "cube": "Sales Plan",
    "rows": 12_400,
    "ms": 412,
    "ok": True,
}

with audit_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(new_record) + "\n")

Each line must be a complete JSON value with no internal newlines, which means `indent=` cannot be used. The single record per line is what makes append, `tail`, `grep`, and split all work without a real parser.

Reading streams one record at a time, so memory stays flat regardless of file size:

In [ ]:
with audit_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        # process record

The alternative, a single top level array (`[{...}, {...}, ...]`), forces the writer to rewrite the whole file on every append and forces the reader to parse the whole file before seeing the first record. For anything that grows, JSON Lines is almost always the better choice.

## 12. Real world design principles

**Use the standard library by default.** The `json` module covers configs, exports, imports, and audit logs without help. Third party libraries like `orjson` and `ujson` are faster and handle more types natively, but they pull in a compiled dependency; reach for them only when profiling shows JSON encode or decode as a real cost.

**Always pass `encoding="utf-8"` to `open`.** The file encoding default is platform dependent, and the failure mode is a config file that works on the developer machine and breaks in a Linux container with a different locale. UTF 8 is the only encoding the JSON specification expects on disk.

**Make committed JSON files stable.** For any JSON file that lives in version control, pass `indent=2`, `sort_keys=True`, and `ensure_ascii=False`. The byte for byte stability makes diffs meaningful and makes accidental changes obvious in code review.

**Validate the shape, do not trust it.** `json.load` returns whatever was in the file. The keys and value types it actually contains are a runtime question. For anything beyond a one off script, validate the shape with a small dataclass conversion in `object_hook` or with a schema library, so the failure mode is a clear error at parse time rather than a `KeyError` three call frames later.

**Decide on a serialization policy once per project.** The set of types a project needs to push through JSON (datetime, Decimal, dataclass) is small and stable. Pick a single `default` function and use it everywhere. Ad hoc lambdas at every call site drift apart over time and produce subtly different outputs.

**Prefer JSON Lines for anything that grows.** Audit logs, run records, exports that another job will tail. A single top level array forces a full rewrite on every append; JSON Lines makes the append the cheapest possible operation.

**Never assume a tuple stays a tuple.** Code that reads data back from JSON must treat tuple input and list input as interchangeable. If the distinction matters downstream, restore it explicitly after `json.loads` rather than hoping the format preserved it.

## 13. Common mistakes

**Opening a JSON file without an explicit encoding.**

In [ ]:
# Wrong
with open("config.json") as f:
    config = json.load(f)            # uses the platform default encoding

# Correct
with open("config.json", encoding="utf-8") as f:
    config = json.load(f)

**Indexing a missing optional key.**

In [ ]:
# Wrong
notify = config["notify"]            # KeyError if "notify" is absent

# Correct
notify = config.get("notify")        # None if absent

**Treating tuples as round trip safe.**

In [ ]:
# Wrong
keys = [(2026, "Jan"), (2026, "Feb")]
loaded = json.loads(json.dumps(keys))
loaded[0] == (2026, "Jan")           # False; loaded[0] is [2026, "Jan"]

# Correct
keys = [(2026, "Jan"), (2026, "Feb")]
loaded = [tuple(k) for k in json.loads(json.dumps(keys))]

**Trying to serialize a dict with non string keys.**

In [ ]:
# Wrong
cells = {(2026, "Jan"): 120_000.0, (2026, "Feb"): 135_000.0}
json.dumps(cells)
# TypeError: keys must be str, int, float, bool or None, not tuple

# Correct
records = [{"year": y, "period": p, "value": v} for (y, p), v in cells.items()]
json.dumps(records)

**Using `default=str` as a catch all.**

In [ ]:
# Wrong
json.dumps(payload, default=str)     # silently stringifies anything unknown

# Correct
def to_json(value: object) -> object:
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")
json.dumps(payload, default=to_json)

**Pretty printing JSON Lines.**

In [ ]:
# Wrong
with open("audit.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, indent=2) + "\n")   # internal newlines break the format

# Correct
with open("audit.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

**Storing an append only log as a single JSON array.**

In [ ]:
# Wrong  (audit.json is rewritten in full on every append)
records = json.loads(Path("audit.json").read_text(encoding="utf-8"))
records.append(new_record)
Path("audit.json").write_text(json.dumps(records), encoding="utf-8")

# Correct  (audit.jsonl appends are O(1))
with Path("audit.jsonl").open("a", encoding="utf-8") as f:
    f.write(json.dumps(new_record) + "\n")